# 04 · Case study — Екатеринбург (EKB)

**Пайплайн НИР — case study.** Применение NBCO к реальной сети общественного
транспорта Екатеринбурга (не бенчмарк):

1. загрузка сети + статистика + гео-рендер исходных маршрутов;
2. прогон Our NBCO (GNN rebuild + trim/extend) с alpha-свипом;
3. сохранение лучшего решения (минимальная стоимость).

Основные эксперименты статьи (E1/E2/MACSA) — в `03_nbco_experiments.ipynb`.

> Логика — в `data.EKBDataSource` и config-first раннере; ячейки только вызывают.
> EKB крупнее бенчмарков (67 маршрутов) — прогон заметно дольше, чем Mandl.

In [ ]:
import pandas as pd
from types import SimpleNamespace
from IPython.display import display

from connectpt.routes_generator.data import EKBDataSource, as_route_tensor
from connectpt.routes_generator.paper_experiments.paper_runs import run_experiment, save_results
from connectpt.routes_generator import render_report
from connectpt.routes_generator.reports import (
    network_connectivity_stats, route_stats, project_coords)
from connectpt.routes_generator.reports.ekb import EKB_COORD_CRS

## Куда пишем и сколько считаем

Те же два параметра, что у `scripts/run_nbco.py`: `--out-dir` (папка
результатов) и `-p n_iterations` (бюджет BCO). `ITERS=None` — бюджет из YAML
(50 итераций на точку).

In [ ]:
# Куда писать результаты и с каким бюджетом (то же, что --out-dir / -p n_iterations
# у scripts/run_nbco.py). ITERS=None -> бюджет из YAML эксперимента (50 итераций).
OUT_DIR = 'artifacts/results/notebook_04_ekb'
ITERS = 2      # короткий прогон; для полного кейса поставьте None
print('результаты ->', OUT_DIR, '| итераций BCO:', ITERS or 'из конфига')

## Сеть EKB: загрузка, статистика, гео-рендер исходных маршрутов

`EKBDataSource().load()` даёт `Instance` (тензоры + исходные маршруты + гео-координаты).

In [ ]:
instance = EKBDataSource().load()
stats = route_stats(instance.init_routes)
latlon = project_coords(instance.coords, EKB_COORD_CRS)
conn = network_connectivity_stats(instance.tensors, instance.init_routes)

print('spec        :', instance.spec)
print('маршруты    :', stats)
print('координаты  :', tuple(instance.coords.shape), '| CRS', EKB_COORD_CRS)
print('связность   :', {'компонент': conn['n_components'],
      'изолир. узлов': conn['isolated_nodes'], 'симметрич.': conn['symmetric'],
      'спрос между компонентами %': round(conn['cross_component_demand_pct'], 4)})

seed_report = SimpleNamespace(
    run_name='EKB seed', table=pd.DataFrame(),
    routes={'Исходные маршруты EKB': instance.init_routes},
    instance=instance, metadata={})
for fig in render_report(seed_report, kind='gis', title='EKB — исходная сеть').figures.values():
    display(fig)

## Our NBCO на EKB (GNN rebuild + trim/extend) + alpha-свип

Config-first прогон с гео-рендером результата. Бюджет — из `ITERS`; сетка alpha
берётся из `cfg/experiments/ekb_case_study.yaml`.

In [ ]:
ekb = run_experiment('ekb_case_study', out_dir=OUT_DIR, kind='gis', title='EKB NBCO', n_iterations=ITERS)
ekb.display()

### Таблица alpha-свипа

In [ ]:
display(ekb.table)

## Лучшее решение (минимальная стоимость)

Строка свипа с минимальным `cost` сохраняется под фиксированным стемом `ekb_best`
(исходные + улучшенные маршруты, координаты, уличный граф).

In [ ]:
art = ekb.artifact
if not art.table.empty:
    best = art.table.loc[art.table['cost'].idxmin()]
    key = f"{best['method']} a={best['alpha']} t={best['adj_target']}"
    best_row = art.table[art.table['alpha'] == best['alpha']].round(6)
    save_results(
        'ekb_best', out_dir=OUT_DIR, table=best_row,
        routes={'Исходные маршруты EKB': as_route_tensor(art.instance.init_routes),
                key: as_route_tensor(art.routes[key])},
        coords=art.instance.coords, street_adj=art.instance.street_adj,
        meta={'city': 'EKB', 'best_method': key})
    display(best_row)
    print('[EKB best] сохранено -> ekb_best (метод:', key, ')')
else:
    print('[EKB best] пусто — свип не дал строк')